# Evaluating results of Deepseek word sense disambiguation

In [15]:
import pandas as pd
import json
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

In [49]:
rus_annotations = pd.read_csv('../gold_data/rus_gold.csv')
rus_queries = pd.read_json('../deepseek_answers/deepseek_rus_results.json')

# load sentences to exclude repeating sentences(only in russian)
rus_sentences = list(rus_queries['sentence'])
# load descriptions to exclude cases with one definition
rus_descriptions = list(rus_queries['descriptions'])
# get gold annotations
rus_gold = list(rus_annotations['selected_option'].str.split(';').str[0])
# get deepseek answers
rus_deepseek_answers = list(rus_queries['senseIDs'])

In [50]:
eng_annotations = pd.read_csv('../gold_data/eng_gold.csv')
eng_queries = pd.read_json('../deepseek_answers/deepseek_eng_results.json')

eng_sentences = list(eng_queries['sentence'])
eng_descriptions = list(eng_queries['descriptions'])
eng_gold = list(eng_annotations['selected_option'].str.split(';').str[0])
eng_deepseek_answers = list(eng_queries['sense_IDs'])

In [51]:
kyr_annotations = pd.read_csv('../gold_data/kyr_gold.csv')
kyr_queries = pd.read_json('../deepseek_answers/deepseek_kyr_results.json')

kyr_sentences = list(kyr_queries['sentence'])
kyr_descriptions = list(kyr_queries['descriptions'])
kyr_gold = list(kyr_annotations['selected_option'])
kyr_deepseek_answers = list(kyr_queries['senseIDs'])

In [53]:
def evaluate_results(sentences, descriptions_list, gold, deepseek_answers, use_glosses=False, skip_one_def=True):
    # right = 0
    all_ans = 0
    # multiple_ans = 0
    sentence_set = set()

    y_true, y_pred = [], []

    for sentence, descriptions, right_ans, predicted_answers in zip(sentences, descriptions_list, gold, deepseek_answers):
        # check case with repeating sentence
        if sentence in sentence_set:
            continue
        sentence_set.add(sentence)

        # skipping words with one definition, can be turned off
        if (skip_one_def and len(descriptions) == 1) or right_ans == 'Нет верного варианта' or right_ans == 'Спорный случай':
            continue

        all_ans += 1

        predicted_ans = predicted_answers if isinstance(predicted_answers, list) else [predicted_answers]

        if use_glosses:
            y_true.append(descriptions[right_ans])
            y_pred.append(descriptions[predicted_ans[0]] if len(predicted_ans) > 0 else None)
        else:
            y_true.append(right_ans)
            y_pred.append(predicted_ans[0] if len(predicted_ans) > 0 else None)
            
        # if len(deepseek_ans) >= 1 and right_ans == deepseek_ans[0]:
        #     if len(deepseek_ans) > 1:
        #         multiple_ans += 1
        #     right += 1

    macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )

    print(f"Macro-P: {macro_prec:.4f}, Macro-R: {macro_rec:.4f}, Macro-F1: {macro_f1:.4f}")

    # Micro‑метрики (будут равны accuracy)
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {accuracy:.4f}")

In [54]:
evaluate_results(rus_sentences, rus_descriptions, rus_gold, rus_deepseek_answers)

Macro-P: 0.6144, Macro-R: 0.6169, Macro-F1: 0.6058
Accuracy: 0.8891


In [55]:
evaluate_results(eng_sentences, eng_descriptions, eng_gold, eng_deepseek_answers)

Macro-P: 0.6025, Macro-R: 0.5686, Macro-F1: 0.5649
Accuracy: 0.7465


In [56]:
evaluate_results(kyr_sentences, kyr_descriptions, kyr_gold, kyr_deepseek_answers, use_glosses=True)

Macro-P: 0.7134, Macro-R: 0.6879, Macro-F1: 0.6894
Accuracy: 0.9406
